# Hybrid CNN + Vision Transformer Autoencoder for Image Denoising

This notebook demonstrates how to design, train, and evaluate a **Hybrid CNN + Vision Transformer (ViT) Autoencoder** for removing additive Gaussian noise from handwritten digits (MNIST dataset) using TensorFlow and Keras.

### Key Technical Features:
1. **CNN Encoder**: Uses standard convolutional layers combined with **Residual Blocks** to extract local spatial features.
2. **Vision Transformer Bottleneck**: Implements a custom **PatchEmbedding**, **PositionalEncoding**, and **TransformerEncoderBlock** from scratch (using standard Keras MultiHeadAttention and Dense layers) to capture global relational contexts across the feature grid.
3. **CNN Decoder with Skip Connections**: Reconstructs the original spatial grid from the bottleneck representation using skip connections from the encoder stages, preserving high-frequency edges and structural shapes.

## 1. Imports and Environmental Configuration

We first import all necessary libraries and our modular code implementations, and then configure seeds for reproducibility.

In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Ensure project files can be loaded from the local path
sys.path.append(os.path.abspath('.'))

from config import Config
from dataset import load_data, get_train_val_test_splits, add_gaussian_noise, print_dataset_statistics
from model import build_model, ResidualBlock2D, PatchEmbedding, PositionalEncoding, TransformerEncoderBlock, HybridAutoencoder
from utils import calculate_metrics, plot_denoising_results, plot_loss_curves

ImportError: cannot import name 'Config' from 'config' (unknown location)

In [ ]:
# Setup output directories and seeds
Config.setup_environment()
print(f"Random seed initialized: {Config.SEED}")
print(f"Device configurations verified.")

## 2. Dataset Loading and Noise Injection

We load the MNIST dataset (with auto-fallback to Keras datasets if Kaggle download fails), normalize the pixel intensities to `[0.0, 1.0]`, split into Train/Validation/Test subsets, and add Gaussian noise with a configurable `NOISE_FACTOR`.

In [ ]:
# Load full datasets
(x_train_full, y_train_full), (x_test_full, y_test_full) = load_data()

# Split into training, validation, and test subsets
x_train, y_train, x_val, y_val, x_test, y_test = get_train_val_test_splits(
    x_train_full, y_train_full, x_test_full, y_test_full, 
    val_split=Config.VAL_SPLIT, seed=Config.SEED
)

# Pre-generate noisy subsets
x_train_noisy = add_gaussian_noise(x_train, noise_factor=Config.NOISE_FACTOR, seed=Config.SEED)
x_val_noisy = add_gaussian_noise(x_val, noise_factor=Config.NOISE_FACTOR, seed=Config.SEED + 1)
x_test_noisy = add_gaussian_noise(x_test, noise_factor=Config.NOISE_FACTOR, seed=Config.SEED + 2)

In [ ]:
# Display statistics
print_dataset_statistics("Clean Train Set", x_train)
print_dataset_statistics("Noisy Train Set", x_train_noisy)

In [ ]:
# Visualize first 5 samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i in range(5):
    axes[0, i].imshow(x_train[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title("Original Clean", loc='left', fontsize=10, fontweight='bold')
        
    axes[1, i].imshow(x_train_noisy[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title("Noisy Input", loc='left', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Build the Hybrid Model Architecture

We compile the model, showing the full parameter breakdown. Note how the Conv feature map downsamples to $7 \times 7 \times 128$, which then gets flattened into $49$ sequence tokens of $128$-dimensional vector embeddings before entering the Transformer Bottleneck blocks.

In [ ]:
model = build_model(Config)
model.summary()

## 4. Model Training

We build the input pipelines with optimal caching and prefetching. We will check if a pre-trained model is already available on disk. If so, we load it directly to save time; otherwise, we initiate training.

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train_noisy, x_train))
train_ds = train_ds.shuffle(10000).batch(Config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val_noisy, x_val))
val_ds = val_ds.batch(Config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

model_path = Config.SAVED_MODEL_PATH

if os.path.exists(model_path):
    print(f"[INFO] Found pre-trained model checkpoint at {model_path}. Loading weights...")
    custom_objects = {
        "ResidualBlock2D": ResidualBlock2D,
        "PatchEmbedding": PatchEmbedding,
        "PositionalEncoding": PositionalEncoding,
        "TransformerEncoderBlock": TransformerEncoderBlock,
        "HybridAutoencoder": HybridAutoencoder
    }
    model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
    print("Model loaded successfully!")
else:
    print("[INFO] No pre-trained model checkpoint found. Training model for 20 epochs...")
    checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
        filepath=model_path, monitor='val_loss', save_best_only=True, verbose=1
    )
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    )
    lr_scheduler_cb = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    )
    
    history = model.fit(
        train_ds,
        epochs=20,
        validation_data=val_ds,
        callbacks=[checkpoint_cb, early_stopping_cb, lr_scheduler_cb]
    )
    
    # Plot loss curves inline
    plt.figure(figsize=(8, 4.5))
    plt.plot(history.history['loss'], 'b-', label='Train Loss', linewidth=2)
    plt.plot(history.history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    plt.title('Training & Validation Loss curves', fontsize=12, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()

## 5. Model Evaluation and Quantitative Metrics

We evaluate the model on the test subset and calculate quantitative image reconstruction metrics: Mean Squared Error (MSE), Peak Signal-to-Noise Ratio (PSNR), and Structural Similarity Index (SSIM).

In [ ]:
print("[INFO] Evaluating model on the test dataset...")
x_test_reconstructed = model.predict(x_test_noisy, batch_size=Config.BATCH_SIZE, verbose=1)

metrics = calculate_metrics(x_test, x_test_reconstructed)

print("\n" + "="*40)
print("       QUANTITATIVE TEST METRICS")
print("="*40)
print(f"Mean Squared Error (MSE) : {metrics['mse']:.6f}")
print(f"Peak Signal-to-Noise Ratio (PSNR) : {metrics['psnr']:.4f} dB")
print(f"Structural Similarity Index (SSIM): {metrics['ssim']:.4f}")
print("="*40 + "\n")

## 6. Qualitative Visual Comparison and Error Heatmaps

Finally, we extract random test digits, run denoising, and display a complete side-by-side reconstruction grid along with an absolute pixel difference heatmap $|Original - Reconstructed|$.

In [ ]:
# Select 5 random index samples
rng = np.random.default_rng(Config.SEED)
indices = rng.choice(len(x_test), size=5, replace=False)

clean_samples = x_test[indices]
noisy_samples = x_test_noisy[indices]
reconstructed_samples = x_test_reconstructed[indices]
diff_samples = np.abs(clean_samples - reconstructed_samples)

fig, axes = plt.subplots(4, 5, figsize=(11, 8.5))
for i in range(5):
    # Clean Image
    axes[0, i].imshow(clean_samples[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title("Original", fontsize=10, fontweight='bold')
        
    # Noisy Image
    axes[1, i].imshow(noisy_samples[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title("Noisy Input", fontsize=10, fontweight='bold')
        
    # Reconstructed Image
    axes[2, i].imshow(reconstructed_samples[i].squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_title("Reconstructed", fontsize=10, fontweight='bold')
        
    # Error Heatmap
    im = axes[3, i].imshow(diff_samples[i].squeeze(), cmap='inferno')
    axes[3, i].axis('off')
    if i == 0:
        axes[3, i].set_title("Diff Heatmap", fontsize=10, fontweight='bold')
        
fig.subplots_adjust(right=0.88, wspace=0.1, hspace=0.25)
cbar_ax = fig.add_axes([0.91, 0.08, 0.02, 0.18])
fig.colorbar(im, cax=cbar_ax)
cbar_ax.set_title("Error", fontsize=9, fontweight='bold')
plt.show()